# 1. Step up and installation

### Evaluation metric

Kaggle uses the **Brier score** (mean squared error between predicted probability and outcome):

$$\text{Brier} = \frac{1}{N}\sum_{i=1}^{N}(p_i - y_i)^2$$

Unlike log-loss, the Brier score does **not** explode for confident wrong predictions — the maximum penalty per game is 1.0. This makes extreme predictions (e.g., 0.99 for a 1-seed vs 16-seed) less risky than under log-loss. Nonetheless, we still clip predictions to [0.01, 0.99] and invest in calibration (Section 8) to minimize overall error.

In [1]:
# Python libraries 
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd 
import os

# Machine learning libraries
# XGBoost
from xgboost import XGBClassifier, XGBRegressor
# Light GBM 
from lightgbm import LGBMClassifier, LGBMRegressor
# Brier score loss 
from sklearn.metrics import brier_score_loss




# 2. Load Dataset 


I load **Detailed Results** files (box scores) for both men's and women's basketball.

| File | Contents | Available from |
|------|----------|---------------|
| `MRegularSeasonDetailedResults` | Every men's regular-season game with full box scores | 2003 |
| `WRegularSeasonDetailedResults` | Every women's regular-season game with full box scores | 2010 |
| `M/WNCAATourneyDetailedResults` | Tournament games with full box scores | same |
| `M/WNCAATourneySeeds` | Seedings (1–16) for each tournament team | same |

**Box score columns** include FGM/FGA (field goals made/attempted), FGM3/FGA3 (three-pointers), FTM/FTA (free throws), OR/DR (offensive/defensive rebounds), Ast (assists), TO (turnovers), Stl (steals), Blk (blocks), PF (personal fouls).

Data is current through early February 2026. The remaining regular-season weeks will be added before Selection Sunday (March 15).



In [2]:
# Locate directory 
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/march-machine-learning-mania-2026/Conferences.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WNCAATourneyDetailedResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WRegularSeasonCompactResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MNCAATourneySeedRoundSlots.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MRegularSeasonDetailedResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MNCAATourneyCompactResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MGameCities.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WSecondaryTourneyCompactResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WGameCities.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MSeasons.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WNCAATourneySlots.csv
/kaggle/input/competitions/march-machine-learning

In [3]:
# Load all the dataset 
directPath = '/kaggle/input/competitions/march-machine-learning-mania-2026'
dataframes = {}

for file in os.listdir(directPath):
    if file.endswith(".csv"):
        name = file.replace(".csv", "")
        dataframes[name] = pd.read_csv(os.path.join(directPath, file))

print(dataframes.keys())



dict_keys(['Conferences', 'WNCAATourneyDetailedResults', 'WRegularSeasonCompactResults', 'MNCAATourneySeedRoundSlots', 'MRegularSeasonDetailedResults', 'MNCAATourneyCompactResults', 'MGameCities', 'WSecondaryTourneyCompactResults', 'WGameCities', 'MSeasons', 'WNCAATourneySlots', 'MSecondaryTourneyTeams', 'SampleSubmissionStage2', 'Cities', 'MTeamSpellings', 'MRegularSeasonCompactResults', 'MMasseyOrdinals', 'MSecondaryTourneyCompactResults', 'WTeams', 'WConferenceTourneyGames', 'MNCAATourneySlots', 'MNCAATourneySeeds', 'WNCAATourneyCompactResults', 'WSeasons', 'WNCAATourneySeeds', 'MTeamCoaches', 'MConferenceTourneyGames', 'WRegularSeasonDetailedResults', 'MNCAATourneyDetailedResults', 'WTeamSpellings', 'MTeamConferences', 'MTeams', 'WTeamConferences', 'SampleSubmissionStage1', 'WSecondaryTourneyTeams'])


In [4]:
#  Men's basketball
mensregularResults = pd.read_csv(f"{directPath}/MRegularSeasonDetailedResults.csv")
menstourneyResults = pd.read_csv(f"{directPath}/MNCAATourneyDetailedResults.csv")
menSeeds = pd.read_csv(f"{directPath}/MNCAATourneySeeds.csv")

# Women's basketball
womensregularResults = pd.read_csv(f"{directPath}/WRegularSeasonDetailedResults.csv")
womenstourneyResults = pd.read_csv(f"{directPath}/WNCAATourneyDetailedResults.csv")
womenSeeds = pd.read_csv(f"{directPath}/WNCAATourneySeeds.csv")

# 3. Data 

In [5]:
mensregularResults  = pd.read_csv(f"{directPath}/MRegularSeasonDetailedResults.csv")

# View the first 5 rows
mensregularResults.head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,10,1104,68,1328,62,N,0,27,58,...,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,70,1393,63,N,0,26,62,...,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,73,1437,61,N,0,24,58,...,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,56,1457,50,N,0,18,38,...,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,77,1208,71,N,0,30,61,...,16,17,27,21,15,12,10,7,1,14


In [6]:
menstourneyResults = pd.read_csv(f"{directPath}/MNCAATourneyDetailedResults.csv")

# View the first 5 rows
menstourneyResults.head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,134,1421,92,1411,84,N,1,32,69,...,31,14,31,17,28,16,15,5,0,22
1,2003,136,1112,80,1436,51,N,0,31,66,...,16,7,7,8,26,12,17,10,3,15
2,2003,136,1113,84,1272,71,N,0,31,59,...,28,14,21,20,22,11,12,2,5,18
3,2003,136,1141,79,1166,73,N,0,29,53,...,17,12,17,14,17,20,21,6,6,21
4,2003,136,1143,76,1301,74,N,1,27,64,...,21,15,20,10,26,16,14,5,8,19


In [7]:
menSeeds = pd.read_csv(f"{directPath}/MNCAATourneySeeds.csv")
# View the first 5 rows
menSeeds.head()

,Season,Seed,TeamID
0,1985,W01,1207
1,1985,W02,1210
2,1985,W03,1228
3,1985,W04,1260
4,1985,W05,1374


In [8]:
womensregularResults  = pd.read_csv(f"{directPath}/WRegularSeasonDetailedResults.csv")

# View the first 5 rows
womensregularResults.head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2010,11,3103,63,3237,49,H,0,23,54,...,13,6,10,11,27,11,23,7,6,19
1,2010,11,3104,73,3399,68,N,0,26,62,...,21,14,27,14,26,7,20,4,2,27
2,2010,11,3110,71,3224,59,A,0,29,62,...,14,19,23,17,23,8,15,6,0,15
3,2010,11,3111,63,3267,58,A,0,27,52,...,26,16,25,22,22,15,11,14,5,14
4,2010,11,3119,74,3447,70,H,1,30,74,...,17,11,21,21,32,12,14,4,2,14


In [9]:
womenstourneyResults = pd.read_csv(f"{directPath}/WNCAATourneyDetailedResults.csv")

# View the first 5 rows
womenstourneyResults.head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2010,138,3124,69,3201,55,N,0,28,57,...,34,3,5,17,19,12,18,4,1,18
1,2010,138,3173,67,3395,66,N,0,23,59,...,27,14,15,18,26,8,8,8,6,22
2,2010,138,3181,72,3214,37,H,0,26,57,...,15,3,8,10,21,4,16,6,4,20
3,2010,138,3199,75,3256,61,H,0,25,63,...,20,17,22,16,21,13,16,5,4,24
4,2010,138,3207,62,3265,42,N,0,24,68,...,26,11,17,16,22,9,10,3,4,12


In [10]:
womensSeeds = pd.read_csv(f"{directPath}/WNCAATourneySeeds.csv")

# View the first 5 rows
womensSeeds.head()

,Season,Seed,TeamID
0,1998,W01,3330
1,1998,W02,3163
2,1998,W03,3112
3,1998,W04,3301
4,1998,W05,3272
